In [1]:
import os
import json
import numpy as np
from pathlib import Path

from mus_align.types import *

In [10]:
def get_alignment(imagepath, audiopath, bboxes_json_path):
    from mus_align.types import Image
    image = Image.from_path(imagepath)
    bboxes = []
    with open(bboxes_json_path, "r") as f:
        bboxes_json = json.load(f)
    for line in bboxes_json:
        for bbox in line:
            bboxes.append(BoundingBox.from_json(bbox))
    measures = [Measure(bbox=bbox) for bbox in bboxes]
    page = Page(measures=measures,image=image)
    audio = Audio.from_path(audiopath)    
    num_measures = len(measures)
    audio_len = audio.length_seconds
    times = [i*(audio_len/num_measures) for i in range(num_measures)] + [audio_len]
    measure_indices = [i for i in range(num_measures)] + [num_measures]
    from PIL import Image
    image = Image.open(imagepath)
    image.save("temp_image.pdf", "PDF", resolution=100.0)
    logical_order = [(page, measure) for measure in measures]
    pdf = PDF.from_path("temp_image.pdf")
    score = Score(score_pages=[page], logical_order=logical_order, score_pdf=pdf)
    
    alignment = MeasureAwareAlignment(
        measure_indices=measure_indices,
        times=times,
        logical_order=score.logical_order,
        max_time=audio_len,
    )

    alignedscore = AlignedScore(
        score_pages=score.score_pages,
        alignment=alignment,
        performance=audio,
        score_pdf=score.score_pdf,
    )

    filepath= f"../approx_alignments/{imagepath[3:-4]}.zip"
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "wb") as f:
        f.write(AlignedScore.as_project_zip(alignedscore))

In [11]:
imagepath = "../images/ex.png"
audiopath = "../audio/ex.mp3"
bboxes_json = "../detected_bboxes/images/ex_bboxes.json"

In [12]:
get_alignment(imagepath, audiopath, bboxes_json)